# 🦺 An Toàn Lao Động — Deep Learning với YOLOv8
### Dataset: HardHat-Vest Dataset v3 (Kaggle)

---

## 📌 Quy trình 7 bước Deep Learning

| Bước | Nội dung |
|:----:|----------|
| **B1** | Chuẩn bị & Khảo sát dữ liệu (EDA) |
| **B2** | Tiền xử lý & Augmentation |
| **B3** | Xây dựng & Cấu hình mô hình YOLOv8 |
| **B4** | Huấn luyện mô hình (Training) |
| **B5** | Đánh giá mô hình (Evaluation) |
| **B6** | Visualize kết quả |
| **B7** | Xuất & Lưu mô hình (Export & Save) |

---
**Lưu ý Kaggle:** Vào `Session options` (góc phải) → `Accelerator` → **GPU T4 x2** → Save

## ⚙️ Cài đặt môi trường

In [ ]:
!pip install ultralytics -q

import os, shutil, yaml, random, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import cv2
from pathlib import Path
from PIL import Image
from IPython.display import display, Image as IPImage
from ultralytics import YOLO
import torch

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

print('=' * 50)
print('✅ Môi trường sẵn sàng')
print(f'   PyTorch  : {torch.__version__}')
print(f'   CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU      : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print('=' * 50)

---
# BƯỚC 1 — Chuẩn bị & Khảo sát dữ liệu (EDA)
> Mục tiêu: Hiểu rõ dataset trước khi đưa vào mô hình

In [ ]:
# ── Đường dẫn dataset trên Kaggle ────────────────────────────────────
DATASET_PATH = Path('/kaggle/input/datasets/muhammetzahitaydn/hardhat-vest-dataset-v3')
WORKING_DIR  = Path('/kaggle/working')  # thư mục lưu output

# Cấu trúc thực tế của dataset này:
# images/train/, images/val/, images/test/
# labels/train/, labels/val/, labels/test/
TRAIN_IMG = DATASET_PATH / 'images' / 'train'
TRAIN_LBL = DATASET_PATH / 'labels' / 'train'
VAL_IMG   = DATASET_PATH / 'images' / 'val'
VAL_LBL   = DATASET_PATH / 'labels' / 'val'
TEST_IMG  = DATASET_PATH / 'images' / 'test'
TEST_LBL  = DATASET_PATH / 'labels' / 'test'

# Kiểm tra đường dẫn
print('📁 Kiểm tra cấu trúc dataset:')
for name, path in [
    ('Train images', TRAIN_IMG), ('Train labels', TRAIN_LBL),
    ('Val   images', VAL_IMG),   ('Val   labels', VAL_LBL),
    ('Test  images', TEST_IMG),  ('Test  labels', TEST_LBL),
]:
    exists = '✅' if path.exists() else '❌'
    count  = len(list(path.glob('*'))) if path.exists() else 0
    print(f'   {exists} {name}: {path} ({count} files)')

In [ ]:
# ── Đọc class names từ classes.txt ───────────────────────────────────
classes_txt = DATASET_PATH / 'labels' / 'classes.txt'

with open(classes_txt, 'r') as f:
    CLASS_NAMES = [line.strip() for line in f.readlines() if line.strip()]

NUM_CLASSES = len(CLASS_NAMES)

print(f'📋 Số lượng classes : {NUM_CLASSES}')
print(f'   Tên classes      : {CLASS_NAMES}')
print()

# Đếm ảnh mỗi split
n_train = len(list(TRAIN_IMG.glob('*.jpg')) + list(TRAIN_IMG.glob('*.png')))
n_val   = len(list(VAL_IMG.glob('*.jpg'))   + list(VAL_IMG.glob('*.png')))
n_test  = len(list(TEST_IMG.glob('*.jpg'))  + list(TEST_IMG.glob('*.png')))

print(f'📊 Số lượng ảnh:')
print(f'   Train : {n_train}')
print(f'   Val   : {n_val}')
print(f'   Test  : {n_test}')
print(f'   Tổng  : {n_train + n_val + n_test}')

In [ ]:
# ── Thống kê số lượng object mỗi class ───────────────────────────────
def count_labels(label_dir):
    counts = {i: 0 for i in range(NUM_CLASSES)}
    for lf in Path(label_dir).glob('*.txt'):
        with open(lf) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cls_id = int(parts[0])
                    if cls_id in counts:
                        counts[cls_id] += 1
    return counts

train_counts = count_labels(TRAIN_LBL)
val_counts   = count_labels(VAL_LBL)

stats_df = pd.DataFrame({
    'Class'     : CLASS_NAMES,
    'Train'     : [train_counts[i] for i in range(NUM_CLASSES)],
    'Validation': [val_counts[i]   for i in range(NUM_CLASSES)],
})
stats_df['Total']   = stats_df['Train'] + stats_df['Validation']
stats_df['Train %'] = (stats_df['Train'] / stats_df['Total'] * 100).round(1)

print('📊 Thống kê số lượng object theo class:')
display(stats_df)

colors = ['#2196F3', '#4CAF50', '#F44336', '#FF9800']
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Phân phối nhãn trong Dataset', fontsize=14, fontweight='bold')

x = np.arange(NUM_CLASSES)
w = 0.35
axes[0].bar(x - w/2, stats_df['Train'],      w, label='Train',      color=colors[0], alpha=0.85)
axes[0].bar(x + w/2, stats_df['Validation'], w, label='Validation', color=colors[1], alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(stats_df['Class'], rotation=20, ha='right')
axes[0].set_title('Số lượng object mỗi class')
axes[0].set_ylabel('Số lượng')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
for i, (t, v) in enumerate(zip(stats_df['Train'], stats_df['Validation'])):
    axes[0].text(i - w/2, t + 5, str(t), ha='center', va='bottom', fontsize=9)
    axes[0].text(i + w/2, v + 5, str(v), ha='center', va='bottom', fontsize=9)

axes[1].pie(stats_df['Total'], labels=stats_df['Class'],
            colors=colors[:NUM_CLASSES], autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 10})
axes[1].set_title('Tỷ lệ phân phối tổng thể')

plt.tight_layout()
plt.savefig('/kaggle/working/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu: class_distribution.png')

In [ ]:
# ── Phân tích kích thước bounding box ────────────────────────────────
def analyze_bboxes(label_dir):
    widths, heights, areas, classes = [], [], [], []
    for lf in Path(label_dir).glob('*.txt'):
        with open(lf) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls = int(parts[0])
                    cx, cy, w, h = map(float, parts[1:])
                    widths.append(w)
                    heights.append(h)
                    areas.append(w * h)
                    classes.append(CLASS_NAMES[cls])
    return pd.DataFrame({'width': widths, 'height': heights, 'area': areas, 'class': classes})

bbox_df = analyze_bboxes(TRAIN_LBL)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('Phân tích Bounding Box (Train set)', fontsize=13, fontweight='bold')

axes[0].hist(bbox_df['width'],  bins=40, color='#2196F3', alpha=0.8, edgecolor='white')
axes[0].set_title('Phân phối chiều rộng BBox')
axes[0].set_xlabel('Width (0–1)')
axes[0].axvline(bbox_df['width'].mean(), color='red', linestyle='--',
                label=f'Mean={bbox_df["width"].mean():.3f}')
axes[0].legend()

axes[1].hist(bbox_df['height'], bins=40, color='#4CAF50', alpha=0.8, edgecolor='white')
axes[1].set_title('Phân phối chiều cao BBox')
axes[1].set_xlabel('Height (0–1)')
axes[1].axvline(bbox_df['height'].mean(), color='red', linestyle='--',
                label=f'Mean={bbox_df["height"].mean():.3f}')
axes[1].legend()

for i, cls in enumerate(CLASS_NAMES):
    sub = bbox_df[bbox_df['class'] == cls]
    axes[2].scatter(sub['width'], sub['height'], alpha=0.3, s=10,
                    color=colors[i % len(colors)], label=cls)
axes[2].set_title('Width vs Height theo Class')
axes[2].set_xlabel('Width')
axes[2].set_ylabel('Height')
axes[2].legend(markerscale=2, fontsize=8)

plt.tight_layout()
plt.savefig('/kaggle/working/bbox_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 Thống kê BBox:')
print(bbox_df[['width','height','area']].describe().round(4))

In [ ]:
# ── Xem mẫu ảnh có annotation ────────────────────────────────────────
def visualize_samples(img_dir, lbl_dir, n=6):
    img_files = random.sample(list(Path(img_dir).glob('*.jpg')), min(n, 50))
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle('Mẫu ảnh với Bounding Box annotation', fontsize=13, fontweight='bold')

    cls_colors = {'hardhat':'#2196F3','vest':'#4CAF50','no-hardhat':'#F44336','no-vest':'#FF9800'}

    for ax, img_file in zip(axes.flatten(), img_files):
        img = np.array(Image.open(img_file).convert('RGB'))
        h, w = img.shape[:2]
        lbl_file = Path(lbl_dir) / (img_file.stem + '.txt')
        ax.imshow(img)
        if lbl_file.exists():
            with open(lbl_file) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cls_id = int(parts[0])
                        cx, cy, bw, bh = map(float, parts[1:])
                        x1 = int((cx - bw/2) * w)
                        y1 = int((cy - bh/2) * h)
                        cls_name = CLASS_NAMES[cls_id]
                        color = cls_colors.get(cls_name, '#9C27B0')
                        rect = patches.Rectangle((x1,y1), int(bw*w), int(bh*h),
                                                  linewidth=2, edgecolor=color, facecolor='none')
                        ax.add_patch(rect)
                        ax.text(x1, y1-4, cls_name, color='white', fontsize=7,
                                bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.85))
        ax.set_title(img_file.name[:25], fontsize=8)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('/kaggle/working/sample_annotations.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_samples(TRAIN_IMG, TRAIN_LBL)
print('✅ Đã lưu: sample_annotations.png')

---
# BƯỚC 2 — Tiền xử lý & Augmentation
> Mục tiêu: Tạo data.yaml chuẩn, kiểm tra dataset, cấu hình augmentation

In [ ]:
# ── Kiểm tra tính toàn vẹn dataset ───────────────────────────────────
def check_dataset_integrity(img_dir, lbl_dir, split_name):
    img_files = list(Path(img_dir).glob('*.jpg')) + list(Path(img_dir).glob('*.png'))
    errors, no_label = [], []

    for img_file in img_files:
        try:
            img = Image.open(img_file)
            img.verify()
        except Exception:
            errors.append(img_file.name)
            continue
        lbl_file = Path(lbl_dir) / (img_file.stem + '.txt')
        if not lbl_file.exists():
            no_label.append(img_file.name)

    print(f'📋 [{split_name}] Tổng: {len(img_files)} | Lỗi: {len(errors)} | Thiếu label: {len(no_label)}')
    if errors:    print(f'   Ảnh lỗi     : {errors[:3]}')
    if no_label:  print(f'   Thiếu label : {no_label[:3]}')

print('🔍 Kiểm tra tính toàn vẹn dataset...')
check_dataset_integrity(TRAIN_IMG, TRAIN_LBL, 'Train')
check_dataset_integrity(VAL_IMG,   VAL_LBL,   'Val')
print('✅ Kiểm tra hoàn tất')

In [ ]:
# ── Tạo file data.yaml chuẩn cho YOLOv8 ──────────────────────────────
# Phải tạo trong /kaggle/working vì /kaggle/input là read-only
yaml_out = WORKING_DIR / 'data.yaml'

data_yaml_content = f"""path: {DATASET_PATH}
train: images/train
val: images/val
test: images/test
nc: {NUM_CLASSES}
names: {CLASS_NAMES}
"""

with open(yaml_out, 'w') as f:
    f.write(data_yaml_content)

print('✅ Đã tạo data.yaml:')
print(data_yaml_content)

# Xác nhận đường dẫn tồn tại
print('🔍 Kiểm tra đường dẫn trong data.yaml:')
for split in ['images/train', 'images/val', 'images/test']:
    p = DATASET_PATH / split
    n = len(list(p.glob('*.jpg')) + list(p.glob('*.png')))
    status = '✅' if p.exists() else '❌'
    print(f'   {status} {split}: {n} ảnh')

In [ ]:
# ── Cấu hình Augmentation ─────────────────────────────────────────────
AUGMENT_CONFIG = {
    'fliplr'    : 0.5,
    'flipud'    : 0.0,
    'degrees'   : 0.0,
    'translate' : 0.1,
    'scale'     : 0.5,
    'shear'     : 0.0,
    'hsv_h'     : 0.015,
    'hsv_s'     : 0.7,
    'hsv_v'     : 0.4,
    'mosaic'    : 1.0,
    'mixup'     : 0.0,
    'copy_paste': 0.0,
}

print('🔧 Cấu hình Augmentation:')
for k, v in AUGMENT_CONFIG.items():
    print(f'   {k:<15}: {v}')

# Minh họa augmentation
sample_img = random.choice(list(TRAIN_IMG.glob('*.jpg')))
orig = cv2.cvtColor(cv2.imread(str(sample_img)), cv2.COLOR_BGR2RGB)
orig = cv2.resize(orig, (320, 320))

aug_imgs = [
    ('Gốc',       orig),
    ('Lật ngang', np.fliplr(orig)),
    ('Tối hơn',   np.clip(orig * 0.6, 0, 255).astype(np.uint8)),
    ('Sáng hơn',  np.clip(orig * 1.4, 0, 255).astype(np.uint8)),
    ('Xoay 15°',  np.array(Image.fromarray(orig).rotate(15))),
    ('Xoay -15°', np.array(Image.fromarray(orig).rotate(-15))),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Minh họa Data Augmentation', fontsize=13, fontweight='bold')
for ax, (title, img) in zip(axes.flatten(), aug_imgs):
    ax.imshow(img)
    ax.set_title(title, fontsize=11)
    ax.axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/augmentation_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu: augmentation_demo.png')

---
# BƯỚC 3 — Xây dựng & Cấu hình mô hình YOLOv8
> Mục tiêu: Lựa chọn kiến trúc phù hợp

In [ ]:
# ── So sánh các phiên bản YOLOv8 ─────────────────────────────────────
model_variants = pd.DataFrame({
    'Model'     : ['YOLOv8n', 'YOLOv8s', 'YOLOv8m', 'YOLOv8l', 'YOLOv8x'],
    'Tham số'   : ['3.2M', '11.2M', '25.9M', '43.7M', '68.2M'],
    'mAP50-95'  : ['37.3', '44.9', '50.2', '52.9', '53.9'],
    'Tốc độ GPU': ['~1ms', '~2ms', '~5ms', '~8ms', '~14ms'],
    'Phù hợp'   : ['Demo/Edge', 'Cân bằng ✅', 'Production', 'Chính xác cao', 'Research'],
})
print('📊 So sánh các phiên bản YOLOv8:')
display(model_variants)
print('\n🎯 Khuyến nghị: yolov8s (cân bằng tốc độ / độ chính xác)')

In [ ]:
# ── Load mô hình ─────────────────────────────────────────────────────
MODEL_SIZE = 's'          # n=nano, s=small, m=medium
MODEL_NAME = f'yolov8{MODEL_SIZE}.pt'

model = YOLO(MODEL_NAME)

print(f'✅ Đã load mô hình: {MODEL_NAME}')
print(f'   Số tham số : {sum(p.numel() for p in model.model.parameters()):,}')
print()
print('🏗️  Kiến trúc YOLOv8:')
print('   Backbone : CSPDarknet  — Trích xuất đặc trưng')
print('   Neck     : C2f + SPPF  — Kết hợp đặc trưng đa tỷ lệ')
print('   Head     : Decoupled   — Dự đoán bbox + class')

---
# BƯỚC 4 — Huấn luyện mô hình (Training)
> Mục tiêu: Fine-tune YOLOv8 với dataset HardHat-Vest

In [ ]:
# ── Cấu hình Training tối ưu cho Kaggle GPU ───────────────────────────
TRAIN_CONFIG = {
    'data'         : str(yaml_out),
    'epochs'       : 50,
    'imgsz'        : 416,
    'batch'        : 32,
    'device'       : 0,

    'optimizer'    : 'AdamW',
    'lr0'          : 0.01,
    'lrf'          : 0.01,
    'momentum'     : 0.937,
    'weight_decay' : 0.0005,
    'warmup_epochs': 2,

    'cache'        : 'disk',  # cache vào disk, không lo tràn RAM
    'workers'      : 2,
    'amp'          : True,    # Mixed Precision — bắt buộc bật
    'patience'     : 15,      # early stopping

    **AUGMENT_CONFIG,

    'project'      : '/kaggle/working/runs/train',
    'name'         : 'hardhat_vest_v1',
    'save_period'  : 10,      # lưu checkpoint mỗi 10 epochs
    'pretrained'   : True,
    'plots'        : True,
    'verbose'      : True,
}

print('🚀 Cấu hình Training:')
for k, v in TRAIN_CONFIG.items():
    if k not in AUGMENT_CONFIG:
        print(f'   {k:<20}: {v}')

In [ ]:
# ── BẮT ĐẦU TRAINING ─────────────────────────────────────────────────
# ⚠️ Kaggle tự động lưu file trong /kaggle/working/ — không lo mất!
print('🏋️  Bắt đầu huấn luyện YOLOv8...')
print('   Ước tính: ~1–1.5 giờ với Kaggle GPU')
print('   Kaggle tự lưu checkpoint — không lo bị ngắt!')
print('=' * 60)

results = model.train(**TRAIN_CONFIG)

print('\n' + '=' * 60)
print('✅ Training hoàn tất!')
print(f'   Kết quả : /kaggle/working/runs/train/hardhat_vest_v1/')
print(f'   Best    : /kaggle/working/runs/train/hardhat_vest_v1/weights/best.pt')

In [ ]:
# ── Training Curves ───────────────────────────────────────────────────
RESULT_DIR = Path('/kaggle/working/runs/train/hardhat_vest_v1')
csv_path   = RESULT_DIR / 'results.csv'

if csv_path.exists():
    df_results = pd.read_csv(csv_path)
    df_results.columns = df_results.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Training Curves — YOLOv8 HardHat-Vest', fontsize=14, fontweight='bold')

    metrics = [
        ('train/box_loss',       'Box Loss (Train)',   '#F44336'),
        ('train/cls_loss',       'Class Loss (Train)', '#2196F3'),
        ('val/box_loss',         'Box Loss (Val)',     '#FF9800'),
        ('metrics/mAP50(B)',     'mAP@50',             '#4CAF50'),
        ('metrics/precision(B)', 'Precision',          '#9C27B0'),
        ('metrics/recall(B)',    'Recall',             '#00BCD4'),
    ]

    for ax, (col, title, color) in zip(axes.flatten(), metrics):
        if col in df_results.columns:
            ax.plot(df_results['epoch'], df_results[col], color=color, linewidth=2)
            ax.fill_between(df_results['epoch'], df_results[col], alpha=0.15, color=color)
            ax.set_title(title, fontweight='bold')
            ax.set_xlabel('Epoch')
            ax.grid(alpha=0.3)
            best_val = df_results[col].max() if any(k in col for k in ['mAP','precision','recall']) \
                       else df_results[col].min()
            ax.axhline(best_val, color=color, linestyle='--', alpha=0.5,
                       label=f'Best: {best_val:.4f}')
            ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Đã lưu: training_curves.png')

---
# BƯỚC 5 — Đánh giá mô hình (Evaluation)
> Mục tiêu: Đo lường hiệu năng thực sự trên tập val/test

In [ ]:
# ── Load best model & validation ─────────────────────────────────────
BEST_MODEL_PATH = '/kaggle/working/runs/train/hardhat_vest_v1/weights/best.pt'
best_model = YOLO(BEST_MODEL_PATH)

print('📊 Đánh giá trên tập Validation...')
val_metrics = best_model.val(
    data=str(yaml_out),
    split='val',
    imgsz=416,
    batch=32,
    conf=0.25,
    iou=0.5,
    plots=True,
    verbose=True,
)

print('\n' + '=' * 60)
print('📈 KẾT QUẢ ĐÁNH GIÁ:')
print(f'   mAP@50       : {val_metrics.box.map50:.4f}')
print(f'   mAP@50-95    : {val_metrics.box.map:.4f}')
print(f'   Precision    : {val_metrics.box.mp:.4f}')
print(f'   Recall       : {val_metrics.box.mr:.4f}')
print(f'   F1-Score     : {2*val_metrics.box.mp*val_metrics.box.mr/(val_metrics.box.mp+val_metrics.box.mr+1e-8):.4f}')
print('=' * 60)

In [ ]:
# ── Metrics chi tiết theo class ─────────────────────────────────────
import numpy as np

# Flatten để tránh lỗi shape không khớp
p    = np.array(val_metrics.box.p).flatten()
r    = np.array(val_metrics.box.r).flatten()
ap50 = np.array(val_metrics.box.ap50).flatten()
ap   = np.array(val_metrics.box.ap).flatten()
n    = min(len(CLASS_NAMES), len(p), len(r), len(ap50), len(ap))

class_metrics_df = pd.DataFrame({
    'Class'    : CLASS_NAMES[:n],
    'Precision': p[:n].round(4),
    'Recall'   : r[:n].round(4),
    'mAP50'    : ap50[:n].round(4),
    'mAP50-95' : ap[:n].round(4),
})

print('📊 Metrics từng class:')
display(class_metrics_df)

# Bar chart
colors = ['#2196F3', '#4CAF50', '#F44336', '#FF9800']
x, w = np.arange(n), 0.25
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w, class_metrics_df['Precision'], w, label='Precision', color='#2196F3', alpha=0.85)
ax.bar(x,     class_metrics_df['Recall'],    w, label='Recall',    color='#4CAF50', alpha=0.85)
ax.bar(x + w, class_metrics_df['mAP50'],     w, label='mAP50',    color='#FF9800', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES[:n], rotation=20, ha='right')
ax.set_ylim(0, 1.1)
ax.set_title('Precision / Recall / mAP50 theo Class')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

# Tổng hợp
summary = pd.DataFrame({
    'Metric'  : ['mAP@50', 'mAP@50-95', 'Precision', 'Recall', 'F1-Score'],
    'Score'   : [
        val_metrics.box.map50, val_metrics.box.map,
        val_metrics.box.mp,    val_metrics.box.mr,
        2*val_metrics.box.mp*val_metrics.box.mr/(val_metrics.box.mp+val_metrics.box.mr+1e-8)
    ]
})
summary['Score']    = summary['Score'].round(4)
summary['Đánh giá'] = summary['Score'].apply(
    lambda x: '🟢 Tốt' if x >= 0.7 else ('🟡 Trung bình' if x >= 0.5 else '🔴 Cần cải thiện')
)
print('\n🏆 TỔNG HỢP KẾT QUẢ:')
display(summary)


In [ ]:
# ── Confusion Matrix & PR Curve (từ YOLOv8 val) ───────────────────────
val_result_dir = Path('/kaggle/working/runs/train/hardhat_vest_v1')

for fname, title in [
    ('confusion_matrix_normalized.png', 'Confusion Matrix'),
    ('PR_curve.png',                    'Precision-Recall Curve'),
    ('F1_curve.png',                    'F1-Confidence Curve'),
]:
    p = val_result_dir / fname
    if p.exists():
        print(f'📊 {title}')
        display(IPImage(str(p)))

---
# BƯỚC 6 — Visualize kết quả
> Mục tiêu: Quan sát trực quan kết quả detection

In [ ]:
# ── Inference trên ảnh test ───────────────────────────────────────────
test_imgs = list(TEST_IMG.glob('*.jpg'))[:6]
if not test_imgs:
    test_imgs = list(VAL_IMG.glob('*.jpg'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Kết quả Detection — YOLOv8 HardHat-Vest', fontsize=14, fontweight='bold')

for ax, img_path in zip(axes.flatten(), test_imgs):
    img_bgr  = cv2.imread(str(img_path))
    result   = best_model(img_bgr, conf=0.35, verbose=False)[0]
    annotated = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
    ax.imshow(annotated)
    ax.set_title(f'{img_path.name[:20]} | {len(result.boxes)} det', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/detection_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu: detection_results.png')

In [ ]:
# ── Phân tích confidence theo class ──────────────────────────────────
all_confs  = {cls: [] for cls in CLASS_NAMES}
all_counts = {cls: 0  for cls in CLASS_NAMES}

sample_imgs = list(VAL_IMG.glob('*.jpg'))[:100]
for img_path in sample_imgs:
    result = best_model(str(img_path), conf=0.1, verbose=False)[0]
    for box in result.boxes:
        cls_name = CLASS_NAMES[int(box.cls)]
        all_confs[cls_name].append(float(box.conf))
        all_counts[cls_name] += 1

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Phân tích Confidence Score (100 ảnh Val)', fontsize=13, fontweight='bold')

for i, cls in enumerate(CLASS_NAMES):
    if all_confs[cls]:
        axes[0].hist(all_confs[cls], bins=20, alpha=0.6,
                     label=f'{cls} (n={len(all_confs[cls])})',
                     color=colors[i % len(colors)])
axes[0].axvline(0.5, color='red', linestyle='--', linewidth=2, label='Threshold=0.5')
axes[0].set_xlabel('Confidence Score')
axes[0].set_ylabel('Số lượng')
axes[0].set_title('Phân phối Confidence theo Class')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

axes[1].bar(all_counts.keys(), all_counts.values(),
            color=[colors[i] for i in range(len(all_counts))], alpha=0.85)
axes[1].set_title('Tổng Detection theo Class')
axes[1].set_ylabel('Số lượng')
axes[1].grid(axis='y', alpha=0.3)
for i, (cls, cnt) in enumerate(all_counts.items()):
    axes[1].text(i, cnt + 0.5, str(cnt), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('/kaggle/working/confidence_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu: confidence_analysis.png')

In [ ]:
# ── So sánh ảnh gốc và kết quả detection ─────────────────────────────
sample_img = str(random.choice(list(VAL_IMG.glob('*.jpg'))))
result     = best_model(sample_img, conf=0.1, verbose=False)[0]
img_rgb    = cv2.cvtColor(cv2.imread(sample_img), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('So sánh ảnh gốc và kết quả Detection', fontsize=13, fontweight='bold')
axes[0].imshow(img_rgb)
axes[0].set_title('Ảnh gốc')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Kết quả (conf≥0.1) — {len(result.boxes)} objects')
axes[1].axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/detection_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📋 Chi tiết detection:')
for box in result.boxes:
    cls_name = CLASS_NAMES[int(box.cls)]
    conf_val = float(box.conf)
    status   = '❌ VI PHẠM' if 'no-' in cls_name else '✅ Tuân thủ'
    print(f'   {status} | {cls_name:<15} | conf={conf_val:.3f}')

---
# BƯỚC 7 — Xuất & Lưu mô hình (Export & Save)
> Mục tiêu: Lưu vào /kaggle/working/ → tải về từ tab Output

In [ ]:
# ── Copy model về /kaggle/working/ ───────────────────────────────────
from datetime import datetime

SAVE_DIR  = Path('/kaggle/working/saved_models')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M')

best_src = Path('/kaggle/working/runs/train/hardhat_vest_v1/weights/best.pt')
last_src = Path('/kaggle/working/runs/train/hardhat_vest_v1/weights/last.pt')
best_dst = SAVE_DIR / f'best_{timestamp}.pt'
last_dst = SAVE_DIR / f'last_{timestamp}.pt'

for src, dst in [(best_src, best_dst), (last_src, last_dst)]:
    if src.exists():
        shutil.copy(src, dst)
        print(f'✅ {src.name} → {dst} ({src.stat().st_size/1e6:.1f} MB)')

# Nén toàn bộ kết quả training
zip_path = SAVE_DIR / f'training_results_{timestamp}'
shutil.make_archive(str(zip_path), 'zip', '/kaggle/working/runs/train/hardhat_vest_v1')
print(f'✅ Đã nén toàn bộ kết quả → {zip_path}.zip')

print(f'\n📁 Files trong /kaggle/working/saved_models:')
for f in sorted(SAVE_DIR.glob('*')):
    print(f'   {f.name} ({f.stat().st_size/1e6:.1f} MB)')

In [ ]:
# ── Export sang ONNX ──────────────────────────────────────────────────
print('📦 Export sang ONNX...')
try:
    onnx_path = best_model.export(format='onnx', opset=12)
    shutil.copy(str(onnx_path), SAVE_DIR / f'best_{timestamp}.onnx')
    print(f'✅ ONNX đã lưu → {SAVE_DIR}/best_{timestamp}.onnx')
except Exception as e:
    print(f'⚠️  Lỗi export ONNX: {e}')

In [ ]:
# ── Lưu metadata ─────────────────────────────────────────────────────

# Khai báo lại các biến phòng trường hợp session bị reset
MODEL_SIZE = 's'
MODEL_NAME = f'yolov8{MODEL_SIZE}.pt'

model_metadata = {
    'model_name'   : f'YOLOv8{MODEL_SIZE} — HardHat-Vest Detector',
    'version'      : 'v1.0',
    'created_at'   : timestamp,
    'base_model'   : MODEL_NAME,
    'dataset'      : 'HardHat-Vest Dataset v3 (Kaggle)',
    'classes'      : CLASS_NAMES,
    'num_classes'  : NUM_CLASSES,
    'input_size'   : 416,
    'epochs'       : 50,
    'batch_size'   : 32,
    'optimizer'    : 'AdamW',
    'learning_rate': 0.01,
    'metrics': {
        'mAP50'    : round(float(val_metrics.box.map50), 4),
        'mAP50-95' : round(float(val_metrics.box.map),   4),
        'precision': round(float(val_metrics.box.mp),    4),
        'recall'   : round(float(val_metrics.box.mr),    4),
    },
    'usage': {
        'inference_code'    : "model = YOLO('best.pt')\\nresults = model('image.jpg', conf=0.5)",
        'violation_classes' : ['no-hardhat', 'no-vest'],
        'compliance_classes': ['hardhat', 'vest'],
    }
}

meta_path = SAVE_DIR / f'model_metadata_{timestamp}.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(model_metadata, f, ensure_ascii=False, indent=2)

print('✅ Metadata đã lưu:')
print(json.dumps(model_metadata, ensure_ascii=False, indent=2))


In [ ]:
# ── Kiểm tra model và tổng kết ────────────────────────────────────────
print('🧪 Kiểm tra model vừa lưu...')
loaded_model  = YOLO(str(best_dst))
test_img_path = str(random.choice(list(VAL_IMG.glob('*.jpg'))))
result        = loaded_model(test_img_path, conf=0.4, verbose=False)[0]

plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
plt.title(f'Test model đã lưu — {len(result.boxes)} detections', fontsize=12)
plt.axis('off')
plt.tight_layout()
plt.show()

print('\n' + '=' * 60)
print('🎉 HOÀN THÀNH TOÀN BỘ PIPELINE DEEP LEARNING!')
print('=' * 60)
print()
print('📁 Tất cả file đã lưu trong /kaggle/working/saved_models/')
print('   → Vào tab Output (góc phải) để tải về máy')
print()
for f in sorted(SAVE_DIR.glob('*')):
    print(f'   {f.name:<50} ({f.stat().st_size/1e6:.1f} MB)')
print()
print('🚀 Bước tiếp theo: Copy best.pt vào thư mục app.py → chạy Streamlit!')